# NOTEBOOK 2: FEATURE ENGINEERING

**PURPOSE:** Create decay-based metrics, engagement scores, lifecycle features

**INPUT:** preprocessed_data.csv

**OUTPUT:** engineered_features.csv, X_train.csv, X_test.csv, y_train.csv, y_test.csv

In [72]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Load clean customer data
df = pd.read_csv('preprocessed_data.csv')
print(f"Data loaded: {df.shape[0]:,} customers")

# Separate target from features
target = df['Activated'].copy()
features = df.drop('Activated', axis=1).drop('CustomerID', axis=1)

Data loaded: 4,338 customers


## 2. CREATE RFM RANKINGS (1-5 SCALE)


In [73]:
recency = features['Recency_Days']
frequency = features['Frequency']
monetary = features['CLV']

# Rank each dimension (1-5 scale)
features['Recency_Rank'] = pd.qcut(recency.rank(method='first'), 5, labels=[5,4,3,2,1])
features['Frequency_Rank'] = pd.qcut(frequency.rank(method='first'), 5, labels=[1,2,3,4,5])
features['Monetary_Rank'] = pd.qcut(monetary.rank(method='first'), 5, labels=[1,2,3,4,5])

# Convert to numeric
features['Recency_Rank'] = features['Recency_Rank'].astype(int)
features['Frequency_Rank'] = features['Frequency_Rank'].astype(int)
features['Monetary_Rank'] = features['Monetary_Rank'].astype(int)

# RFM Score (average of three ranks)
features['RFM_Score'] = (features['Recency_Rank'] + features['Frequency_Rank'] + features['Monetary_Rank']) / 3

print("✓ RFM rankings created")

✓ RFM rankings created


## 3. CREATE DECAY-BASED FEATURES (PAPER 1 - CORE INNOVATION)

In [74]:
# Decay Score = (Frequency × Monetary) / (Recency + 1)
features['Decay_Score'] = (frequency * monetary) / (recency + 1)

# Normalize Decay Score to 0-100 scale
decay_min = features['Decay_Score'].min()
decay_max = features['Decay_Score'].max()
features['Decay_Score_Normalized'] = (
    (features['Decay_Score'] - decay_min) / (decay_max - decay_min) * 100
)

# Exponential Decay = e^(-0.01 × recency)
features['Exponential_Decay'] = np.exp(-0.01 * recency)

# Recency-Weighted Frequency
features['Recency_Weighted_Frequency'] = frequency * np.exp(-0.02 * recency)

# Monetary Decay Adjusted
features['Monetary_Decay_Adjusted'] = monetary * np.maximum(0, 1 - recency/365)

print("✓ Decay-based features created (Paper 1)")

✓ Decay-based features created (Paper 1)


## 4. CREATE ENGAGEMENT METRICS

In [75]:
# Normalize RFM to 0-1 for weighted calculation
recency_norm = (recency - recency.min()) / (recency.max() - recency.min())
freq_norm = (frequency - frequency.min()) / (frequency.max() - frequency.min())
monetary_norm = (monetary - monetary.min()) / (monetary.max() - monetary.min())

# Engagement Score: Weighted RFM (40% recency, 30% frequency, 30% monetary)
features['Engagement_Score'] = (
    (1 - recency_norm) * 0.40 +
    freq_norm * 0.30 + 
    monetary_norm * 0.30
) * 100

# Transaction Consistency
features['Transaction_Consistency'] = np.clip(
    1 / (1 + features['Total_Quantity'] / (frequency + 1) * 0.1), 0, 1
)

print("✓ Engagement metrics created")

✓ Engagement metrics created


## 5. CREATE LIFECYCLE STAGE FEATURES (ONE-HOT ENCODED)

In [76]:
def get_lifecycle_stage(row):
    if row['Frequency'] == 1:
        return 'ONE_TIME'
    elif row['Recency_Days'] <= 30:
        return 'ACTIVE'
    elif row['Recency_Days'] <= 90:
        return 'AT_RISK'
    elif row['Recency_Days'] <= 180:
        return 'DORMANT'
    else:
        return 'LOST'

lifecycle = features.apply(get_lifecycle_stage, axis=1)
lifecycle_encoded = pd.get_dummies(lifecycle, prefix='Stage')
features = pd.concat([features, lifecycle_encoded], axis=1)

print("✓ Lifecycle stages created")

✓ Lifecycle stages created


## 6. CREATE TEMPORAL FEATURES

In [77]:
# Purchase Velocity (purchases per month)
features['Purchase_Velocity'] = frequency / (features['Account_Age_Days'] / 30 + 1)

# Recency Percentile
features['Recency_Percentile'] = recency.rank(pct=True) * 100

print("✓ Temporal features created")

✓ Temporal features created


## 7. SCALE CONTINUOUS FEATURES (STANDARDSCALER)

In [ ]:
# Convert dates to numeric (days since epoch)
features['First_Purchase_Date'] = pd.to_datetime(features['First_Purchase_Date']).astype('int64') // 10**9 // 86400
features['Last_Purchase_Date'] = pd.to_datetime(features['Last_Purchase_Date']).astype('int64') // 10**9 // 86400

# Now scale all numeric columns
continuous_cols = features.select_dtypes(include=['int64', 'float64']).columns.tolist()
continuous_cols = [col for col in continuous_cols if not col.startswith('Stage_')]

scaler = StandardScaler()
features[continuous_cols] = scaler.fit_transform(features[continuous_cols])

print("✓ Continuous features scaled (StandardScaler)")

✓ Continuous features scaled (StandardScaler)


## 8. STRATIFIED TRAIN-TEST SPLIT (80/20)

In [81]:
# Stratified split preserves class distribution
X_train, X_test, y_train, y_test = train_test_split(
    features, target,
    test_size=0.2,
    random_state=42,
    stratify=target
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"  Activated: {y_train.sum()} ({y_train.sum()/len(y_train)*100:.1f}%)")
print(f"Test set: {X_test.shape[0]} samples")
print(f"  Activated: {y_test.sum()} ({y_test.sum()/len(y_test)*100:.1f}%)")

Training set: 3470 samples
  Activated: 1175 (33.9%)
Test set: 868 samples
  Activated: 294 (33.9%)


## 9. SAVE ALL OUTPUTS

In [82]:
# Full engineered dataset
df_engineered = pd.concat([features, target], axis=1)
df_engineered.to_csv('engineered_features.csv', index=False)

# Train-test splits
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print(f"✓ Feature engineering complete")
print(f"✓ Files saved: engineered_features.csv, X_train/test.csv, y_train/test.csv")

✓ Feature engineering complete
✓ Files saved: engineered_features.csv, X_train/test.csv, y_train/test.csv
